# AgriFusion AI

## Domain:Multimodal Agriculture Intelligence & Decision Agent

***Project Overview***

Multimodal AI Agriculture Intelligence & Decision Agent is an AI-powered agricultural decision-support system that combines multiple input modalities such as plant images, farmer questions, agricultural documents, and environmental information.

The system uses computer vision, deep learning, natural language processing, retrieval-augmented generation, generative AI, and AI agents to analyze agricultural problems and provide context-aware recommendations to farmers.

The primary objective is to move from simple image-based disease detection toward an intelligent agricultural assistant capable of understanding visual information and farmer questions together.

**Problem Statement**

Farmers often identify crop diseases, pests, and plant health problems manually. This process can be difficult because symptoms may look similar across different diseases, and reliable agricultural information may not always be immediately available.

Traditional crop disease detection systems generally focus only on image classification or object detection. Such systems may identify a disease but do not provide complete explanations, treatment information, prevention methods, or answers to farmer-specific questions.

The core problem is the lack of an integrated agricultural AI system that can understand both visual crop information and natural-language questions while retrieving reliable agricultural knowledge.

***Goal***

The goal is to build a multimodal agricultural intelligence system that can analyze crop images, identify diseases or pests, understand farmer questions, retrieve relevant agricultural knowledge from documents, and generate useful agricultural recommendations.

The system combines computer vision, OpenCV, RAG, NLP, Generative AI, and an AI Agent into a single end-to-end pipeline

### System Architecture
#### Multimodal Input

The system accepts different types of agricultural inputs such as plant leaf images, fruit images, stem images, root images, farmer text questions, voice input, video, soil reports, weather data, crop information, and location information.

**AI Processing Pipeline**

The complete pipeline connects multiple AI technologies:

Image → YOLO → OpenCV → Prediction

Documents → Text Extraction → Chunking → Embeddings → Vector Database → RAG

Farmer Question → NLP → Query Understanding

Retrieved Context + Model Prediction → GenAI

All Information → AI Agent → Final Agricultural Decision

#### Install Required Libraries

The required libraries provide support for image processing, deep learning, NLP, RAG, vector databases, and application development.

In [ ]:
!pip install -q -U \
    roboflow \
    opencv-python-headless \
    "numpy>=2.1,<2.3" \
    chromadb \
    sentence-transformers \
    pypdf \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-text-splitters \
    langgraph

### Import Libraries

The imported libraries are used for image processing, machine learning, data manipulation, document processing, embeddings, and vector search.

In [ ]:
import cv2
import numpy as np
import langchain
import langchain_core
import langgraph
import importlib.metadata

print("OpenCV :", cv2.__version__)
print("NumPy  :", np.__version__)
print("LangChain :", langchain.__version__)
print("LangChain Core :", langchain_core.__version__)

try:
    print("LangGraph :", importlib.metadata.version("langgraph"))
except importlib.metadata.PackageNotFoundError:
    print("LangGraph : Not found")

print("\n✅ Environment ready")

### API Configuration

API keys are required to authenticate the external AI services used in the Multimodal Agriculture AI project. The Roboflow API key is used to access Roboflow resources/models, while the OpenAI API key is used for OpenAI-based Generative AI functionality.

The getpass module allows the user to enter the API key without displaying the key directly in the notebook output. The keys are then stored as environment variables so that other parts of the application can access them without hard-coding the secrets.

***Import Required Libraries***

The os module is used to create and access environment variables. The getpass module provides a secure input prompt for entering API keys.

In [ ]:
import os
import getpass

os.environ["ROBOFLOW_API_KEY"] = getpass.getpass(
    "Enter Roboflow API Key: "
)

os.environ["OPENAI_API_KEY"] = getpass.getpass(
    "Enter OpenAI API Key: "
)

print("✅ API keys loaded")

### Connect to Roboflow
## Initialize Roboflow Client

This step creates a connection between the notebook and Roboflow using the API key stored in the environment variable. The API key is retrieved securely without exposing it directly in the code.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(
    api_key=os.environ["ROBOFLOW_API_KEY"]
)

print("✅ Roboflow connected")

### Dataset Collection
#### Image Dataset

The image dataset contains agricultural images required for crop identification, disease detection, and pest detection. Images can be collected from agricultural datasets and object-detection datasets.

Dataset Categories

The image dataset can contain:

Crop images,
Leaf images,
Fruit images,
Stem images,
Root images,
Disease images,
Pest images,
Healthy crop images.

Deep Learning

Already Pre Trained Model Yolo object detection models

In [ ]:
roboflow_links = [
    {"crop": "Orange", "category": "Disease",
     "workspace": "citrus-leaf-disease",
     "project": "citrus-leaves-diseases-ykhpu"},

    {"crop": "Mango", "category": "Disease",
     "workspace": "mango-leaf-disease",
     "project": "mango-leaf-disease-kguxz"},

    {"crop": "Mango", "category": "Pest",
     "workspace": "project-pjsi8",
     "project": "mango-pests"},

    {"crop": "Watermelon", "category": "Disease",
     "workspace": "watermelon-diseases",
     "project": "watermelon-diseases"},

    {"crop": "Banana", "category": "Disease",
     "workspace": "tdtu-ocamc",
     "project": "banana-diseases-lrf9y"},

    {"crop": "Maize", "category": "Disease",
     "workspace": "detection-of-corn-crop-diseases-using-yolov8",
     "project": "maize-disease-dataset"},

    {"crop": "Maize", "category": "Pest",
     "workspace": "mo-qmyfx",
     "project": "maize-eloeo"},

    {"crop": "Beans", "category": "Disease",
     "workspace": "zhang-wen-hao-itjzq",
     "project": "beans-diseases"},

    {"crop": "Grapes", "category": "Disease",
     "workspace": "poramet-thongvichit",
     "project": "grape-leaf-disease-kf5ll"},

    {"crop": "Rice", "category": "Disease",
     "workspace": "riceleafdiseasedetection",
     "project": "rice-leaf-disease-s1asn"},

    {"crop": "Rice", "category": "Pest",
     "workspace": "rice-pest",
     "project": "rice-i9qwz"},

    {"crop": "Pomegranate", "category": "Disease + Pest",
     "workspace": "krish-verma-fayi1",
     "project": "pomegranate-disease-detection-29ocz"},

    {"crop": "Multiple Crops", "category": "Pest",
     "workspace": "yolo-dv5qa",
     "project": "pest-uruhn-yfh5t"}
]

print("Number of models:", len(roboflow_links))

### OpenCV

## Configuration and Helper Functions

The pipeline defines confidence thresholds, filters unwanted dataset-related labels, identifies disease and pest classes, and provides visualization settings for detected objects.

***Roboflow Model Initialization***

The configured Roboflow API key is used to initialize the Roboflow client so that the application can access the registered crop disease and pest detection models.

***File Upload and Type Detection***

The system allows the user to upload one image or video file. The uploaded file extension is checked to determine whether the input should be processed using the image pipeline or the video pipeline.

***Multi-Model Image Scanning***

For an image input, the system scans the registered agriculture models and collects valid predictions. Each detection records the model used, detected class, confidence score, bounding box, and model object.

***Image Detection and Visualization***

The image predictions are sorted according to confidence score. The highest-performing model is selected for visualization, and OpenCV is used to draw bounding boxes and prediction labels on the original image.

***Video Model Selection***

For video input, the first frame is extracted and analyzed using the registered agriculture models. The detection with the highest confidence is used to select the model for subsequent video processing.

***Video Frame Detection***

The selected Roboflow model is applied to video frames at a defined frame interval. Detected diseases, pests, or other classes are displayed using bounding boxes and confidence scores.

***Processed Video Generation***

The detected frames are written into a new video file while preserving the original video dimensions and frame rate. The system also tracks the total number of processed frames and detections.

***Final Agriculture AI Output***

The final pipeline produces either an annotated image or an annotated video containing the detected agricultural objects, their classes, and confidence scores. This output forms the computer-vision layer of the larger Agriculture AI system.

In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display, Video
from roboflow import Roboflow

# ------------------------------------------------------------
# 1. Configurations & Helpers
# ------------------------------------------------------------
MIN_CONFIDENCE = 50.0
INVALID_KEYWORDS = ["this dataset was exported", "roboflow.com", "exported via"]
DISEASE_KWS = ["disease", "spot", "blight", "mildew", "rust", "virus", "rot", "scab"]
PEST_KWS = ["pest", "insect", "worm", "bug", "aphid"]

def get_detection_color(cls_name):
    cls_lower = str(cls_name).lower()
    if any(w in cls_lower for w in DISEASE_KWS): return (255, 0, 0)
    if any(w in cls_lower for w in PEST_KWS): return (255, 128, 0)
    return (0, 200, 0)

rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))

# ------------------------------------------------------------
# 2. File Upload & Type Detection
# ------------------------------------------------------------
print("📹 Upload exactly ONE file (Image OR Video):")
uploaded = files.upload()
if not uploaded:
    raise ValueError("❌ No file uploaded.")

filename = list(uploaded.keys())[0]
ext = os.path.splitext(filename)[1].lower()

is_video = ext in ['.mp4', '.avi', '.mov', '.mkv']
is_image = ext in ['.jpg', '.jpeg', '.png', '.bmp']

if not is_video and not is_image:
    raise ValueError(f"❌ Unsupported file format '{ext}'. Please upload an image or video.")

print(f"\n✅ Successfully uploaded: {filename}")
if is_image:
    print("🖼️ File type detected: IMAGE")
else:
    print("🎬 File type detected: VIDEO")

# ------------------------------------------------------------
# 3. Model Scanning Function (Used for Images & Video Frame 1)
# ------------------------------------------------------------
def scan_image_all_models(img_path):
    print("\n🔍 Scanning across all agriculture models to find the best match...")
    all_detections = []

    for item in roboflow_links:
        crop, cat = item["crop"], item["category"]
        model_name_full = f"{crop} {cat}"
        print(f"   Analyzing: {model_name_full}...")
        try:
            project = rf.workspace(item["workspace"]).project(item["project"])

            # Load model safely
            model = next((v.model for v in project.versions() if hasattr(v, "model") and v.model), None)
            if not model:
                model = next((v.models()[0] for v in project.versions() if hasattr(v, "models") and v.models()), None)
            if not model:
                continue

            # Run Inference
            preds = model.predict(img_path, confidence=int(MIN_CONFIDENCE), overlap=30).json().get("predictions", [])

            for p in preds:
                conf = p.get("confidence", 0) * 100
                cls_name = p.get("class", "")
                if any(kw in cls_name.lower() for kw in INVALID_KEYWORDS): continue

                all_detections.append({
                    "Model": model_name_full,
                    "Class": cls_name,
                    "Confidence": conf,
                    "BBox": (p["x"], p["y"], p["width"], p["height"]),
                    "Model_Obj": model  # Save model object for later video processing
                })
        except Exception:
            pass # Skip models that fail to load

    return all_detections

# ============================================================
# 4A. PROCESS IMAGE
# ============================================================
if is_image:
    detections = scan_image_all_models(filename)

    if not detections:
        print("\n❌ No valid detections found.")
    else:
        # Sort and show results
        df = pd.DataFrame(detections).sort_values(by="Confidence", ascending=False).reset_index(drop=True)
        df["Conf_Display"] = df["Confidence"].apply(lambda x: f"{x:.1f}%")

        print("\n" + "="*60 + "\n🌾 AGRICULTURE AI RESULT (BEST MATCH)\n" + "="*60)
        display(df[["Model", "Class", "Conf_Display"]])

        # Draw on image
        img = cv2.imread(filename)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_h, img_w = img_rgb.shape[:2]

        scale_factor = max(img_h, img_w) / 800.0
        FONT_SCALE = max(0.4, 0.5 * scale_factor)
        TXT_THICK = max(1, int(1.2 * scale_factor))
        BOX_THICK = max(2, int(2.5 * scale_factor))

        # Only draw detections from the top-performing model
        best_model_name = df.iloc[0]["Model"]
        best_detections = [d for d in detections if d["Model"] == best_model_name]

        for d in best_detections:
            x, y, w, h = d["BBox"]
            cls_name, conf = d["Class"], d["Confidence"]

            x1, y1 = int(x - w/2), int(y - h/2)
            x2, y2 = int(x + w/2), int(y + h/2)
            color = get_detection_color(cls_name)

            # Draw Box
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, BOX_THICK)

            # Draw Text
            label = f"{cls_name} {int(conf)}%"
            (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, TXT_THICK)
            txt_y = max(th + 5, y1)
            txt_x = max(0, min(x1, img_w - tw))
            cv2.rectangle(img_rgb, (txt_x, txt_y - th - 5), (txt_x + tw, txt_y + bl - 5), color, cv2.FILLED)
            cv2.putText(img_rgb, label, (txt_x, txt_y - 5), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, (255, 255, 255), TXT_THICK)

        plt.figure(figsize=(12, 12 * (img_h / img_w)))
        plt.imshow(img_rgb)
        plt.axis("off")
        plt.show()

# ============================================================
# 4B. PROCESS VIDEO
# ============================================================
elif is_video:
    cap = cv2.VideoCapture(filename)
    ret, frame = cap.read()
    if not ret:
        raise ValueError("❌ Could not read video file.")

    # Extract first frame to auto-detect which crop model to use
    temp_first_frame = "temp_first_frame.jpg"
    cv2.imwrite(temp_first_frame, frame)

    print("\n🎬 Analyzing the first frame of the video to auto-select the right AI model...")
    detections = scan_image_all_models(temp_first_frame)

    if not detections:
        print("\n❌ No detections found in the first frame. Cannot determine which model to use.")
    else:
        # Select the best model based on the first frame
        best_detection = max(detections, key=lambda x: x["Confidence"])
        selected_model = best_detection["Model_Obj"]
        best_model_name = best_detection["Model"]

        print(f"\n🌾 Auto-selected best model: {best_model_name} (Confidence: {best_detection['Confidence']:.1f}%)")
        print("\n🚀 Starting full video detection...\n")

        fps = cap.get(cv2.CAP_PROP_FPS)
        width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        output_filename = "agriculture_unified_output.mp4"
        out = cv2.VideoWriter(output_filename, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

        # Reset video back to frame 0
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

        frame_number = 0
        total_det_count = 0
        FRAME_SKIP = 10 # Process every 10th frame

        while True:
            ret, frame = cap.read()
            if not ret: break

            frame_number += 1
            if frame_number % FRAME_SKIP != 0:
                out.write(frame)
                continue

            temp_frame = "temp_frame_vid.jpg"
            cv2.imwrite(temp_frame, frame)

            try:
                preds = selected_model.predict(temp_frame, confidence=40, overlap=30).json().get("predictions", [])

                for pred in preds:
                    conf, cls_name = pred.get("confidence", 0) * 100, pred.get("class", "unknown")
                    x, y, w, h = pred.get("x", 0), pred.get("y", 0), pred.get("width", 0), pred.get("height", 0)

                    x1, y1 = int(x - w/2), int(y - h/2)
                    x2, y2 = int(x + w/2), int(y + h/2)
                    color = get_detection_color(cls_name)

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    label = f"{cls_name} {conf:.1f}%"
                    cv2.putText(frame, label, (x1, max(y1-10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
                    total_det_count += 1
            except:
                pass # Skip failed frames

            out.write(frame)
            if frame_number % 100 == 0:
                print(f"Processed {frame_number}/{total_frames} frames...")

        cap.release()
        out.release()

        print("\n" + "="*60 + "\nVIDEO PROCESSING COMPLETED\n" + "="*60)
        print(f"Total frames processed: {frame_number}")
        print(f"Total detections found: {total_det_count}")

        # FIX: Auto-download the file to prevent truncation limit issues
        print("\n📥 Video is too large to embed. Downloading to your computer now...")
        from google.colab import files
        files.download(output_filename)

### Insights

The model detected **Black Sigatoka (83%)** on the banana leaf, with a **Healthy (70%)** detection also identified.


### 🔗 Integration Fix: Bridge CV Detection → RAG → GenAI

**Problem in the original notebook:** the detection step above produces `detections` / `df`
(a DataFrame of bounding boxes and class names), but the later RAG + GenAI function
`agriculture_ai()` and the LangGraph agent both expect a variable called **`detection_result`**
(a simple dict with `crop`, `condition`, `category`, `confidence`). That variable was never
created, so running the notebook end-to-end would fail with `NameError: name 'detection_result' is not defined`.

The cell below fixes this by automatically building `detection_result` from whatever the
detection step produced (image path via `df`, or video path via `best_detection`), with a
manual fallback if no detection was found. This is what actually **connects** the
Deep Learning / OpenCV stage to the NLP + RAG + GenAI stage.


In [ ]:
# ============================================================
# 🔗 INTEGRATION BRIDGE: Deep Learning/OpenCV detection -> RAG + GenAI
# ============================================================
# The detection cell above produces `df` (image path) or `best_detection` (video path).
# The RAG/GenAI function agriculture_ai() and the LangGraph agent both need a
# `detection_result` dict shaped like: {"crop", "condition", "category", "confidence"}.
# This cell builds that dict automatically so the pipeline runs end-to-end.

def build_detection_result():
    try:
        # --- Case 1: an image was processed (cell above created `df`) ---
        if 'df' in globals() and df is not None and len(df) > 0:
            top = df.iloc[0]
            cls_name = str(top["Class"])
            confidence = float(top["Confidence"]) / 100.0
            model_name = top["Model"] if "Model" in top else cls_name
            crop = str(model_name).split()[0]

            if any(k in cls_name.lower() for k in PEST_KWS):
                category = "Pest"
            elif any(k in cls_name.lower() for k in DISEASE_KWS):
                category = "Disease"
            else:
                category = "Healthy"

            return {
                "crop": crop,
                "condition": cls_name,
                "category": category,
                "confidence": confidence,
            }

        # --- Case 2: a video was processed (cell above created `best_detection`) ---
        if 'best_detection' in globals() and best_detection is not None:
            cls_name = str(best_detection["Class"])
            confidence = float(best_detection["Confidence"]) / 100.0
            crop = str(best_model_name).split()[0] if 'best_model_name' in globals() else cls_name.split()[0]

            if any(k in cls_name.lower() for k in PEST_KWS):
                category = "Pest"
            elif any(k in cls_name.lower() for k in DISEASE_KWS):
                category = "Disease"
            else:
                category = "Healthy"

            return {
                "crop": crop,
                "condition": cls_name,
                "category": category,
                "confidence": confidence,
            }

    except Exception as e:
        print("⚠️ Could not auto-build detection_result from detections:", e)

    return None


detection_result = build_detection_result()

if detection_result is None:
    print("⚠️ No detection output found in memory (run the upload/detection cell first).")
    print("   Falling back to manual example values so the rest of the notebook can still run.")
    detection_result = {
        "crop": "Banana",
        "condition": "Black Sigatoka",
        "category": "Disease",
        "confidence": 0.83,
    }

print("✅ detection_result created:")
print(detection_result)


# RAG



### Upload Agricultural Knowledge Documents

This step allows the user to upload one or more PDF documents containing agricultural knowledge. These documents will be used later as the knowledge source for the RAG system, enabling the application to retrieve relevant information and provide evidence-based responses.

In [ ]:
from google.colab import files

pdf_files = files.upload()

print(
    "PDFs uploaded:",
    list(pdf_files.keys())
)

### Load and Split PDF Documents

This step reads each uploaded PDF using PyPDFLoader and divides the extracted content into smaller overlapping chunks. These chunks make the agricultural documents easier to process, embed, store, and retrieve in the RAG system.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

all_documents = []

for pdf_name in pdf_files.keys():

    print(
        "Reading:",
        pdf_name
    )

    loader = PyPDFLoader(pdf_name)

    pages = loader.load()

    chunks = splitter.split_documents(
        pages
    )

    all_documents.extend(chunks)

print(
    "\nTotal chunks:",
    len(all_documents)
)

## Insights

PDFs are converted into overlapping text chunks, creating the foundation for embedding, vector storage, retrieval, and evidence-based agricultural responses.

### Create Agricultural Embeddings

This step converts the extracted agricultural text chunks into numerical vector representations using the all-MiniLM-L6-v2 Hugging Face embedding model. These embeddings allow the RAG system to compare the meaning of a farmer’s question with the knowledge stored in the uploaded PDFs.


## Store Embeddings in ChromaDB

The generated embeddings and their corresponding document chunks are stored in a persistent Chroma vector database. The database can later be searched to retrieve relevant agricultural information for RAG-based responses.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    persist_directory="./agriculture_db"
)

print("✅ Agriculture RAG database created")

## Insights

Agricultural PDF chunks are converted into semantic embeddings and stored in ChromaDB, creating a persistent knowledge base for relevant information retrieval.

### Define the Agricultural Query

This step defines a farmer-oriented question about banana Black Sigatoka, covering its symptoms, treatment, management, and prevention. The query is used to search the agricultural knowledge stored in the vector database.

### Retrieve Relevant Documents

The similarity_search() method compares the query with the stored document embeddings and retrieves the four most semantically relevant chunks from the agricultural knowledge base.

In [ ]:
query = """
What are the symptoms, treatment,
management and prevention of
banana Black Sigatoka?
"""

documents = vector_store.similarity_search(
    query,
    k=4
)


## Display Retrieved Sources

The retrieved documents are displayed individually so that the relevant agricultural evidence can be inspected before sending the context to the GenAI layer.

In [ ]:
print(
    "Retrieved documents:",
    len(documents)
)

for i, doc in enumerate(documents):

    print(
        f"\n========== SOURCE {i+1} =========="
    )

    print(
        doc.page_content[:1000]
    )

### Insights

The RAG system retrieves the four most relevant agricultural knowledge chunks for Black Sigatoka, providing evidence for the later GenAI response.

## Load Free Local LLM

This step replaces the OpenAI API with a free, locally running Hugging Face model. The google/flan-t5-base model generates text directly in the Colab environment, avoiding API quota and key-related errors.

In [ ]:
# Replacing OpenAI with a free local pipeline using Hugging Face to avoid API quota errors
from transformers import pipeline

try:
    # Load a free, local text generation pipeline
    print("⌛ Loading free local LLM (Hugging Face)...")
    llm_local = pipeline("text2text-generation", model="google/flan-t5-base")
    print("✅ Local GenAI Ready")
except Exception as e:
    print("Could not load local model, setting up fallback summary processor.")
    llm_local = None

## Insights

A free local FLAN-T5 model is loaded for GenAI, eliminating OpenAI API quota dependency and enabling RAG-based response generation directly in Colab.

### Integrate Detection, RAG, and Local GenAI

This function connects the computer vision detection result with the local RAG knowledge base and Hugging Face GenAI model. It retrieves relevant agricultural information based on the detected crop and condition, then generates a farmer-friendly response.

### Detection Context

The function extracts the crop, detected condition, category, and confidence score from the computer vision result. This information forms the initial context for the Agriculture AI report.

### RAG Knowledge Retrieval

A search query is created from the detected crop and condition. The ChromaDB vector store then retrieves the three most relevant agricultural knowledge chunks.

### Context Filtering

The retrieved documents are filtered to prioritize information related to Banana and Sigatoka when available. If no matching information is found, the retrieved document content is used as the context.

### Local GenAI Response

The retrieved agricultural context and farmer's question are passed to the local FLAN-T5 model. The model generates an answer using the retrieved knowledge rather than relying only on general model knowledge.

### Fallback Mechanism

If the local GenAI model is unavailable or inference fails, the function returns the retrieved agricultural information directly. This ensures the pipeline can still provide useful source-based information.

### End-to-End Flow

**Detection Result → Crop & Condition → RAG Retrieval → Relevant Context → Local GenAI → Agriculture AI Report** 🌾🤖


In [ ]:
def agriculture_ai(detection_result, farmer_question):
    crop = detection_result.get("crop", "Crop")
    condition = detection_result.get("condition", "Condition")
    category = detection_result.get("category", "Category")
    confidence = detection_result.get("confidence", 0.0)

    # 1. Query local RAG database
    query = f"{crop} {condition} symptoms control prevention"
    documents = vector_store.similarity_search(query, k=3)

    # 2. Extract facts from retrieved documents
    rag_facts = []
    for doc in documents:
        content = doc.page_content.strip()
        if "Sigatoka" in content or "banana" in content.lower():
            rag_facts.append(content)

    context = "\n".join(rag_facts) if rag_facts else "\n".join([d.page_content[:400] for d in documents])

    # 3. Use local free model to formulate response or fallback to direct extraction
    intro = f"🌾 AGRICULTURE AI REPORT\n=======================\n1. Crop: {crop}\n2. Condition: {condition}\n3. Category: {category}\n4. Confidence: {confidence:.1%}\n\n"

    if llm_local:
        prompt = f"Answer the question using the context. Question: {farmer_question}. Context: {context[:800]}"
        try:
            local_answer = llm_local(prompt, max_length=256)[0]['generated_text']
            return intro + "🤖 AI Answer (Local Model):\n" + local_answer + "\n\n📖 Supporting Source Material Found:\n" + context[:600] + "..."
        except Exception as e:
            pass

    # Fallback to precise extraction if model inference fails
    return intro + "📝 Summary of Retrieved Agriculture Information:\n" + context[:1200] + "..."

## NLP Question Asking

## Define the Farmer's Question

This step captures the farmer's natural-language question. The question is later combined with the detected crop and disease information to retrieve relevant knowledge from the RAG database and generate an AI response.

In [ ]:
farmer_question = """
What are the symptoms of this disease
and how can I control it?
"""

## Insights

The farmer can ask questions in natural language. This input is combined with detection results and RAG knowledge to generate a relevant agriculture-focused response. 🌾🤖

# GENAI

## Generate Final Diagnostic Response

This step executes the complete agriculture_ai() pipeline using the detected crop/disease result and the farmer's question. The function performs RAG retrieval, processes the relevant agricultural context, and generates the final response using the local GenAI model or fallback extraction.

In [ ]:
# Execute the diagnostic assistant using the fallback parser
answer = agriculture_ai(detection_result, farmer_question)
print(answer)

## Insights

The complete pipeline combines AI detection + farmer question + RAG retrieval + local GenAI to produce a crop-specific agricultural response. 🌾🤖

## Agriculture AI Agent

### Define Agent State

This section defines the shared state used by the LangGraph agent. It stores the image detection result, the farmer's question, and the final agricultural response.

### Create Agriculture Agent Node

The agent node receives the current state, sends the detected agricultural condition and farmer question to the `agriculture_ai()` function, and stores the generated answer in the state.

### Build Agent Workflow

LangGraph is used to create the workflow. The Agriculture AI node is configured as the entry point and connects directly to the end of the workflow.

### Compile Agriculture Agent

The workflow is compiled into an executable `agriculture_agent`. This creates the agent layer that coordinates the existing detection, RAG, and GenAI components.



In [ ]:
from typing import TypedDict, Optional

from langgraph.graph import StateGraph, END


class AgricultureState(TypedDict):

    image_result: dict
    farmer_question: str
    final_answer: str


def agriculture_agent_node(
    state: AgricultureState
):

    result = state["image_result"]

    question = state["farmer_question"]

    answer = agriculture_ai(
        result,
        question
    )

    return {
        "final_answer": answer
    }


workflow = StateGraph(
    AgricultureState
)

workflow.add_node(
    "agriculture_ai",
    agriculture_agent_node
)

workflow.set_entry_point(
    "agriculture_ai"
)

workflow.add_edge(
    "agriculture_ai",
    END
)

agriculture_agent = workflow.compile()

print("✅ Agriculture Agent created")

### Insights

LangGraph now orchestrates the Agriculture AI pipeline, connecting the detection result and farmer question to RAG and GenAI for a final agricultural response. 🌾🤖


## Execute Agriculture AI Agent
Invoke the Agent

This step sends the detected image result and the farmer's question into the compiled LangGraph Agriculture Agent. The agent processes the input through the configured Agriculture AI workflow and returns the final answer.

In [ ]:
agent_result = agriculture_agent.invoke({

    "image_result": detection_result,

    "farmer_question": """
    What are the symptoms and
    treatment for this problem?
    """

})

print(
    agent_result["final_answer"]
)

## Insights

The LangGraph agent receives the detected condition and farmer question, then coordinates RAG and GenAI to generate the final crop-specific agricultural response.

---
# 🚀 ENHANCED FEATURES

The sections below add production-grade features to the AgriFusion AI pipeline:
Enhanced NLP, Advanced GenAI, Production Error Handling, Real Agent Decision-Making,
Model Evaluation, RAG Evaluation, Testing, FastAPI, and Streamlit.


### Install Additional Dependencies

These packages support the enhanced NLP, evaluation, API, and UI features added below.


In [ ]:
!pip install -q -U \
    spacy \
    fastapi \
    uvicorn \
    pyngrok \
    streamlit \
    rouge-score \
    nltk \
    scikit-learn

# Download spaCy small English model
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ Additional dependencies installed")


---
## 🧠 Section 1: Enhanced NLP Processing (Farmer Question Understanding)

The original notebook uses a simple hardcoded string as the farmer's question.
This section adds **real NLP processing** with:

- **Intent Classification**: Determines what the farmer is asking about (disease info, treatment, prevention, pest control, general)
- **Entity Extraction**: Identifies crop names, disease names, and symptom keywords from the question
- **Query Expansion**: Enriches the farmer's query with synonyms and related terms for better RAG retrieval
- **Structured Output**: Returns a clean dict with `intent`, `entities`, `expanded_query`, and `original_question`


In [ ]:
import spacy
import re
from collections import Counter

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# ── Intent patterns ──────────────────────────────────────────────────
INTENT_PATTERNS = {
    "treatment": [
        "treat", "cure", "remedy", "fix", "heal", "medicine",
        "fungicide", "pesticide", "spray", "apply", "control",
        "manage", "get rid", "stop", "how to",
    ],
    "prevention": [
        "prevent", "avoid", "protect", "safe", "guard",
        "resistant", "before", "precaution", "stop from",
    ],
    "disease_info": [
        "symptom", "sign", "identify", "look like", "what is",
        "cause", "reason", "why", "diagnose", "detect",
        "affect", "damage", "stage",
    ],
    "pest_control": [
        "pest", "insect", "bug", "worm", "aphid", "mite",
        "larva", "caterpillar", "beetle", "fly", "moth",
        "trap", "repel", "biological control",
    ],
    "general": [
        "tell me", "information", "about", "explain", "describe",
        "overview", "detail", "help",
    ],
}

# ── Known agricultural entities ─────────────────────────────────────
KNOWN_CROPS = {
    "banana", "mango", "rice", "maize", "corn", "wheat",
    "orange", "citrus", "tomato", "potato", "grape", "bean",
    "watermelon", "pomegranate", "apple", "cotton", "soybean",
    "sugarcane", "pepper", "onion", "garlic", "coconut",
}

KNOWN_DISEASES = {
    "sigatoka", "black sigatoka", "yellow sigatoka", "panama disease",
    "blight", "late blight", "early blight", "leaf spot",
    "powdery mildew", "downy mildew", "rust", "leaf rust",
    "stem rust", "anthracnose", "fusarium wilt", "bacterial wilt",
    "mosaic virus", "leaf curl", "root rot", "brown spot",
    "blast", "smut", "scab", "canker", "dieback",
    "cercospora", "alternaria", "septoria",
}

QUERY_EXPANSION_MAP = {
    "treatment": ["management", "chemical control", "organic control", "application"],
    "prevention": ["resistant varieties", "crop rotation", "cultural practices"],
    "disease_info": ["pathogen", "lifecycle", "transmission", "spread"],
    "pest_control": ["integrated pest management", "IPM", "biological control"],
    "general": ["overview", "impact", "economic loss"],
}


def classify_intent(text: str) -> str:
    """Classify the farmer's intent based on keyword matching."""
    text_lower = text.lower()
    scores = {}
    for intent, keywords in INTENT_PATTERNS.items():
        scores[intent] = sum(1 for kw in keywords if kw in text_lower)
    best_intent = max(scores, key=scores.get)
    return best_intent if scores[best_intent] > 0 else "general"


def extract_entities(text: str) -> dict:
    """Extract crop names, disease names, and symptoms from farmer text."""
    doc = nlp(text)
    text_lower = text.lower()

    # Extract known crops
    crops = [c for c in KNOWN_CROPS if c in text_lower]

    # Extract known diseases
    diseases = [d for d in KNOWN_DISEASES if d in text_lower]

    # Extract spaCy named entities as additional context
    spacy_entities = [
        {"text": ent.text, "label": ent.label_}
        for ent in doc.ents
        if ent.label_ in ("ORG", "PRODUCT", "GPE", "DATE", "QUANTITY")
    ]

    # Extract symptom keywords using noun chunks
    symptom_keywords = []
    symptom_terms = {
        "spot", "lesion", "wilt", "yellow", "brown", "black",
        "curl", "rot", "necrosis", "chlorosis", "stunted",
        "drooping", "discolor", "hole", "streak", "ring",
    }
    for token in doc:
        if token.lemma_.lower() in symptom_terms:
            symptom_keywords.append(token.text.lower())

    return {
        "crops": list(set(crops)),
        "diseases": list(set(diseases)),
        "symptoms": list(set(symptom_keywords)),
        "spacy_entities": spacy_entities,
    }


def expand_query(original_query: str, intent: str, entities: dict) -> str:
    """Expand the farmer's query with synonyms and related terms for better RAG retrieval."""
    parts = [original_query.strip()]

    # Add entity context
    if entities["crops"]:
        parts.append(" ".join(entities["crops"]))
    if entities["diseases"]:
        parts.append(" ".join(entities["diseases"]))
    if entities["symptoms"]:
        parts.append("symptoms: " + ", ".join(entities["symptoms"]))

    # Add intent-based expansion terms
    expansion = QUERY_EXPANSION_MAP.get(intent, [])
    if expansion:
        parts.append(" ".join(expansion[:2]))  # Add top-2 expansion terms

    return " ".join(parts)


def process_farmer_question(question: str, detection_ctx: dict = None) -> dict:
    """
    Full NLP pipeline for a farmer's question.

    Parameters
    ----------
    question : str
        The farmer's natural language question.
    detection_ctx : dict, optional
        Detection result dict with 'crop', 'condition', 'category', 'confidence'.

    Returns
    -------
    dict with keys: intent, entities, expanded_query, original_question
    """
    # Classify intent
    intent = classify_intent(question)

    # Extract entities
    entities = extract_entities(question)

    # Enrich entities from detection context if available
    if detection_ctx:
        crop = detection_ctx.get("crop", "")
        condition = detection_ctx.get("condition", "")
        if crop and crop.lower() not in [c.lower() for c in entities["crops"]]:
            entities["crops"].append(crop)
        if condition and condition.lower() not in [d.lower() for d in entities["diseases"]]:
            entities["diseases"].append(condition)

    # Expand query
    expanded = expand_query(question, intent, entities)

    result = {
        "intent": intent,
        "entities": entities,
        "expanded_query": expanded,
        "original_question": question.strip(),
    }

    print("🧠 NLP Analysis:")
    print(f"   Intent       : {intent}")
    print(f"   Crops        : {entities['crops']}")
    print(f"   Diseases     : {entities['diseases']}")
    print(f"   Symptoms     : {entities['symptoms']}")
    print(f"   Expanded Q   : {expanded[:120]}...")

    return result


# ── Demo ──────────────────────────────────────────────────────────────
nlp_result = process_farmer_question(
    farmer_question,
    detection_ctx=detection_result
)
print("\n✅ Enhanced NLP processing ready")


---
## 🤖 Section 2: Enhanced GenAI with Structured Prompts

Upgrades the basic FLAN-T5 prompting to use **structured prompt templates** that produce
multi-section agricultural reports. The prompt is dynamically adjusted based on:
- Detection confidence (high vs. low)
- Detected category (disease vs. pest vs. healthy)
- Farmer's classified intent
- Available RAG context


In [ ]:
# ============================================================
# ENHANCED GENAI WITH STRUCTURED PROMPTS
# ============================================================

PROMPT_TEMPLATE_DISEASE = """You are an expert agricultural advisor.
A crop disease has been detected with the following details:
- Crop: {crop}
- Disease: {condition}
- Confidence: {confidence:.0%}
- Farmer's question intent: {intent}

Use the following agricultural knowledge to answer:
{context}

Farmer's question: {question}

Provide a structured response covering:
1. DIAGNOSIS: Confirm the detected condition
2. SYMPTOMS: Key visible symptoms to look for
3. TREATMENT: Recommended treatment steps
4. PREVENTION: How to prevent recurrence
5. ORGANIC ALTERNATIVES: Natural/organic options if available
"""

PROMPT_TEMPLATE_PEST = """You are an expert agricultural advisor.
A crop pest has been detected with the following details:
- Crop: {crop}
- Pest: {condition}
- Confidence: {confidence:.0%}
- Farmer's question intent: {intent}

Use the following agricultural knowledge to answer:
{context}

Farmer's question: {question}

Provide a structured response covering:
1. IDENTIFICATION: Confirm the pest detected
2. DAMAGE PATTERN: What damage this pest causes
3. CHEMICAL CONTROL: Recommended pesticides/sprays
4. BIOLOGICAL CONTROL: Natural predators or bio-agents
5. INTEGRATED PEST MANAGEMENT: Combined strategy
"""

PROMPT_TEMPLATE_HEALTHY = """You are an expert agricultural advisor.
The crop appears healthy based on AI analysis:
- Crop: {crop}
- Status: Healthy
- Confidence: {confidence:.0%}

Available agricultural knowledge:
{context}

Farmer's question: {question}

Provide advice on:
1. CURRENT STATUS: Confirm the healthy status
2. MAINTENANCE: Best practices to maintain crop health
3. COMMON RISKS: Diseases/pests to watch for
4. NUTRITION: Recommended fertilization schedule
"""

LOW_CONFIDENCE_ADDENDUM = """\n\nIMPORTANT: The detection confidence is below 70%.
Please note this uncertainty in your response and suggest the farmer
verify the diagnosis through physical inspection or local agricultural
extension services."""


def select_prompt_template(category: str) -> str:
    """Select the right prompt template based on detection category."""
    category_lower = category.lower()
    if "pest" in category_lower:
        return PROMPT_TEMPLATE_PEST
    elif "healthy" in category_lower:
        return PROMPT_TEMPLATE_HEALTHY
    else:
        return PROMPT_TEMPLATE_DISEASE


def format_genai_response(raw_text: str, detection_result: dict) -> str:
    """Post-process and format the GenAI response into a clean report."""
    crop = detection_result.get("crop", "Unknown")
    condition = detection_result.get("condition", "Unknown")
    category = detection_result.get("category", "Unknown")
    confidence = detection_result.get("confidence", 0.0)

    header = (
        "\n" + "=" * 60 + "\n"
        "🌾 AGRIFUSION AI — AGRICULTURAL DIAGNOSTIC REPORT\n"
        + "=" * 60 + "\n\n"
        f"📋 Crop       : {crop}\n"
        f"🔬 Condition  : {condition}\n"
        f"📂 Category   : {category}\n"
        f"📊 Confidence : {confidence:.1%}\n"
    )

    if confidence < 0.7:
        header += "⚠️  WARNING: Low confidence detection — verify manually.\n"

    header += "\n" + "-" * 60 + "\n"

    footer = (
        "\n" + "-" * 60 + "\n"
        "💡 Disclaimer: This AI advisory is for guidance only.\n"
        "   Always consult local agricultural extension officers\n"
        "   for region-specific recommendations.\n"
        + "=" * 60
    )

    return header + raw_text.strip() + footer


def agriculture_ai_enhanced(detection_result: dict, farmer_question: str,
                             nlp_result: dict = None) -> str:
    """
    Enhanced agriculture AI function with structured prompts.
    Integrates NLP analysis, category-aware prompts, and confidence-based logic.
    """
    crop = detection_result.get("crop", "Crop")
    condition = detection_result.get("condition", "Condition")
    category = detection_result.get("category", "Category")
    confidence = detection_result.get("confidence", 0.0)

    # Use NLP expanded query if available, else build simple query
    if nlp_result:
        search_query = nlp_result["expanded_query"]
        intent = nlp_result["intent"]
    else:
        search_query = f"{crop} {condition} symptoms control prevention"
        intent = "general"

    # RAG retrieval with enhanced query
    documents = vector_store.similarity_search(search_query, k=4)
    context = "\n\n".join([doc.page_content[:500] for doc in documents])

    # Select and fill prompt template
    template = select_prompt_template(category)
    prompt = template.format(
        crop=crop,
        condition=condition,
        confidence=confidence,
        intent=intent,
        context=context[:800],
        question=farmer_question.strip(),
    )

    if confidence < 0.7:
        prompt += LOW_CONFIDENCE_ADDENDUM

    # Generate response
    if llm_local:
        try:
            raw_answer = llm_local(prompt[:1024], max_length=300)[0]["generated_text"]
        except Exception as e:
            raw_answer = f"[GenAI inference error: {e}]\n\nRetrieved Information:\n{context[:800]}"
    else:
        raw_answer = f"Retrieved Agricultural Information:\n{context[:1000]}"

    # Format and return
    return format_genai_response(raw_answer, detection_result)


# ── Demo ──────────────────────────────────────────────────────────────
enhanced_answer = agriculture_ai_enhanced(
    detection_result, farmer_question, nlp_result
)
print(enhanced_answer)
print("\n✅ Enhanced GenAI ready")


---
## 🛡️ Section 3: Production Error Handling & Logging

Adds structured logging, input validation, retry logic, and graceful degradation
to make the pipeline production-ready. Includes:

- **Structured Logging**: `logging` module with file + console handlers
- **Input Validation**: File type, size, dimensions checks
- **API Key Validation**: Verify keys before making API calls
- **SafePipelineExecutor**: Wraps any function with try/except + retry + timeout
- **Graceful Degradation**: Returns partial results instead of crashing


In [ ]:
import logging
import time
import functools
from pathlib import Path
from typing import Any, Callable

# ============================================================
# 3.1 STRUCTURED LOGGING SETUP
# ============================================================
LOG_FORMAT = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
LOG_DATE_FORMAT = "%Y-%m-%d %H:%M:%S"

def setup_logging(log_level=logging.INFO, log_file="agrifusion.log"):
    """Configure structured logging with both console and file output."""
    logger = logging.getLogger("AgriFusion")
    logger.setLevel(log_level)
    logger.handlers.clear()

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(log_level)
    ch.setFormatter(logging.Formatter(LOG_FORMAT, datefmt=LOG_DATE_FORMAT))
    logger.addHandler(ch)

    # File handler
    fh = logging.FileHandler(log_file, mode="a")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter(LOG_FORMAT, datefmt=LOG_DATE_FORMAT))
    logger.addHandler(fh)

    return logger

logger = setup_logging()
logger.info("AgriFusion AI logging initialized")

# ============================================================
# 3.2 INPUT VALIDATION
# ============================================================
MAX_IMAGE_SIZE_MB = 20
MAX_VIDEO_SIZE_MB = 200
VALID_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VALID_VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
MAX_IMAGE_DIMENSION = 8192  # pixels


def validate_image_file(filepath: str) -> dict:
    """Validate an image file for processing."""
    errors = []
    warnings = []
    path = Path(filepath)

    # Check existence
    if not path.exists():
        errors.append(f"File not found: {filepath}")
        return {"valid": False, "errors": errors, "warnings": warnings}

    # Check extension
    ext = path.suffix.lower()
    if ext not in VALID_IMAGE_EXTS:
        errors.append(f"Invalid image format '{ext}'. Supported: {VALID_IMAGE_EXTS}")

    # Check file size
    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_IMAGE_SIZE_MB:
        errors.append(f"Image too large: {size_mb:.1f}MB (max {MAX_IMAGE_SIZE_MB}MB)")
    elif size_mb > MAX_IMAGE_SIZE_MB * 0.8:
        warnings.append(f"Image is close to size limit: {size_mb:.1f}MB")

    # Check dimensions
    if not errors:
        try:
            img = cv2.imread(filepath)
            if img is None:
                errors.append("Could not decode image file (corrupt or unsupported codec)")
            else:
                h, w = img.shape[:2]
                if h > MAX_IMAGE_DIMENSION or w > MAX_IMAGE_DIMENSION:
                    errors.append(f"Image dimensions {w}x{h} exceed max {MAX_IMAGE_DIMENSION}px")
                if h < 32 or w < 32:
                    errors.append(f"Image dimensions {w}x{h} too small (min 32x32)")
        except Exception as e:
            errors.append(f"Image validation error: {e}")

    valid = len(errors) == 0
    if valid:
        logger.info(f"Image validated: {filepath} ({size_mb:.1f}MB)")
    else:
        logger.error(f"Image validation failed: {errors}")

    return {"valid": valid, "errors": errors, "warnings": warnings}


def validate_video_file(filepath: str) -> dict:
    """Validate a video file for processing."""
    errors = []
    warnings = []
    path = Path(filepath)

    if not path.exists():
        errors.append(f"File not found: {filepath}")
        return {"valid": False, "errors": errors, "warnings": warnings}

    ext = path.suffix.lower()
    if ext not in VALID_VIDEO_EXTS:
        errors.append(f"Invalid video format '{ext}'. Supported: {VALID_VIDEO_EXTS}")

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_VIDEO_SIZE_MB:
        errors.append(f"Video too large: {size_mb:.1f}MB (max {MAX_VIDEO_SIZE_MB}MB)")

    if not errors:
        try:
            cap = cv2.VideoCapture(filepath)
            if not cap.isOpened():
                errors.append("Could not open video file")
            else:
                frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                if frames == 0:
                    errors.append("Video has 0 frames")
            cap.release()
        except Exception as e:
            errors.append(f"Video validation error: {e}")

    valid = len(errors) == 0
    return {"valid": valid, "errors": errors, "warnings": warnings}


def validate_api_keys() -> dict:
    """Check that required API keys are set and non-empty."""
    results = {}
    for key_name in ["ROBOFLOW_API_KEY", "OPENAI_API_KEY"]:
        val = os.environ.get(key_name, "")
        if val and len(val) > 5:
            results[key_name] = "✅ Set"
        elif val:
            results[key_name] = "⚠️ Suspiciously short"
        else:
            results[key_name] = "❌ Missing"
    return results


# ============================================================
# 3.3 SAFE PIPELINE EXECUTOR (retry + error handling)
# ============================================================
def retry_with_backoff(max_retries: int = 3, backoff_factor: float = 1.0):
    """Decorator that retries a function with exponential backoff."""
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs) -> Any:
            last_exception = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    wait = backoff_factor * (2 ** attempt)
                    logger.warning(
                        f"{func.__name__} attempt {attempt+1}/{max_retries} "
                        f"failed: {e}. Retrying in {wait:.1f}s..."
                    )
                    time.sleep(wait)
            logger.error(f"{func.__name__} failed after {max_retries} retries: {last_exception}")
            raise last_exception
        return wrapper
    return decorator


class SafePipelineExecutor:
    """
    Wraps pipeline stages with error handling, logging, and timing.
    Returns partial results on failure instead of crashing.
    """

    def __init__(self):
        self.execution_log = []

    def execute(self, stage_name: str, func: Callable,
                *args, fallback=None, **kwargs) -> Any:
        """Execute a pipeline stage with error handling."""
        start = time.time()
        try:
            logger.info(f"Stage '{stage_name}' starting...")
            result = func(*args, **kwargs)
            elapsed = time.time() - start
            logger.info(f"Stage '{stage_name}' completed in {elapsed:.2f}s")
            self.execution_log.append({
                "stage": stage_name, "status": "success",
                "duration": elapsed
            })
            return result
        except Exception as e:
            elapsed = time.time() - start
            logger.error(f"Stage '{stage_name}' failed after {elapsed:.2f}s: {e}")
            self.execution_log.append({
                "stage": stage_name, "status": "failed",
                "error": str(e), "duration": elapsed
            })
            if fallback is not None:
                logger.info(f"Using fallback for '{stage_name}'")
                return fallback
            raise

    def summary(self) -> str:
        """Return a summary of all executed stages."""
        lines = ["\n📊 Pipeline Execution Summary:", "-" * 40]
        for entry in self.execution_log:
            status_icon = "✅" if entry["status"] == "success" else "❌"
            lines.append(
                f"  {status_icon} {entry['stage']}: "
                f"{entry['duration']:.2f}s"
            )
            if "error" in entry:
                lines.append(f"     Error: {entry['error']}")
        return "\n".join(lines)


# ── Demo ──────────────────────────────────────────────────────────────
executor = SafePipelineExecutor()

print("API Key Status:")
for k, v in validate_api_keys().items():
    print(f"  {k}: {v}")

print("\n✅ Production error handling & logging ready")


---
## 🤖 Section 4: Upgraded LangGraph Agent with Real Decision-Making

Replaces the basic single-node LangGraph agent with a **multi-node workflow** that:

1. **Analyzes** the image detection → classifies severity
2. **Routes conditionally** based on severity (emergency / standard / healthy)
3. **Retrieves** knowledge from RAG with expanded NLP queries
4. **Generates** a structured response
5. **Quality-checks** the response before returning

This is a real **agent** with conditional branching, not just a linear pipeline.


In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import StateGraph, END

# ============================================================
# 4.1 ENHANCED AGENT STATE
# ============================================================
class EnhancedAgricultureState(TypedDict):
    """State shared across all agent nodes."""
    # Inputs
    image_result: dict
    farmer_question: str

    # NLP stage
    nlp_analysis: Optional[dict]

    # Severity classification
    severity: Optional[str]          # "emergency" | "moderate" | "low" | "healthy"
    severity_reasoning: Optional[str]

    # RAG retrieval
    retrieved_context: Optional[str]
    retrieval_scores: Optional[list]

    # Response generation
    draft_response: Optional[str]
    final_answer: Optional[str]

    # Quality check
    quality_passed: Optional[bool]
    quality_feedback: Optional[str]

    # Agent trace
    reasoning_trace: Optional[list]


# ============================================================
# 4.2 AGENT NODE FUNCTIONS
# ============================================================
def node_nlp_analysis(state: EnhancedAgricultureState) -> dict:
    """Node 1: Analyze the farmer's question with NLP."""
    trace = state.get("reasoning_trace", []) or []
    trace.append("[NLP] Analyzing farmer question...")

    nlp_result = process_farmer_question(
        state["farmer_question"],
        detection_ctx=state["image_result"]
    )

    trace.append(f"[NLP] Intent: {nlp_result['intent']}, "
                 f"Entities: {nlp_result['entities']['crops']} + "
                 f"{nlp_result['entities']['diseases']}")

    return {"nlp_analysis": nlp_result, "reasoning_trace": trace}


def node_classify_severity(state: EnhancedAgricultureState) -> dict:
    """Node 2: Classify the severity of the detected condition."""
    trace = state.get("reasoning_trace", []) or []
    det = state["image_result"]
    confidence = det.get("confidence", 0)
    category = det.get("category", "Unknown").lower()
    condition = det.get("condition", "").lower()

    # Severity rules
    emergency_diseases = [
        "panama disease", "fusarium wilt", "bacterial wilt",
        "citrus greening", "fire blight", "late blight",
    ]
    moderate_keywords = [
        "sigatoka", "rust", "mildew", "anthracnose",
        "leaf spot", "blast",
    ]

    if category == "healthy":
        severity = "healthy"
        reasoning = "Crop appears healthy based on detection"
    elif any(d in condition for d in emergency_diseases) and confidence > 0.6:
        severity = "emergency"
        reasoning = f"Highly destructive disease '{det['condition']}' detected with {confidence:.0%} confidence"
    elif any(k in condition for k in moderate_keywords) or confidence > 0.5:
        severity = "moderate"
        reasoning = f"Disease '{det['condition']}' requires attention (confidence: {confidence:.0%})"
    else:
        severity = "low"
        reasoning = f"Low confidence ({confidence:.0%}) or minor condition detected"

    trace.append(f"[SEVERITY] Classified as: {severity.upper()} — {reasoning}")
    return {
        "severity": severity,
        "severity_reasoning": reasoning,
        "reasoning_trace": trace,
    }


def node_rag_retrieval(state: EnhancedAgricultureState) -> dict:
    """Node 3: Retrieve relevant agricultural knowledge."""
    trace = state.get("reasoning_trace", []) or []
    nlp = state.get("nlp_analysis") or {}
    query = nlp.get("expanded_query", state["farmer_question"])

    # Adjust k based on severity
    k = 6 if state.get("severity") == "emergency" else 4
    trace.append(f"[RAG] Retrieving top-{k} documents with expanded query")

    docs = vector_store.similarity_search_with_score(query, k=k)
    context_parts = []
    scores = []
    for doc, score in docs:
        context_parts.append(doc.page_content[:500])
        scores.append(round(float(score), 4))

    context = "\n\n---\n\n".join(context_parts)
    trace.append(f"[RAG] Retrieved {len(docs)} documents, scores: {scores}")

    return {
        "retrieved_context": context,
        "retrieval_scores": scores,
        "reasoning_trace": trace,
    }


def node_emergency_response(state: EnhancedAgricultureState) -> dict:
    """Node 4a: Generate an URGENT response for emergency cases."""
    trace = state.get("reasoning_trace", []) or []
    trace.append("[RESPONSE] Generating EMERGENCY response with urgency markers")

    det = state["image_result"]
    context = state.get("retrieved_context", "")

    prompt = (
        f"URGENT agricultural alert. {det['condition']} detected on {det['crop']} "
        f"with {det['confidence']:.0%} confidence. This is a severe disease. "
        f"Context: {context[:600]}. "
        f"Question: {state['farmer_question']}. "
        f"Provide immediate action steps."
    )

    if llm_local:
        try:
            raw = llm_local(prompt[:1024], max_length=300)[0]["generated_text"]
        except Exception:
            raw = context[:800]
    else:
        raw = context[:800]

    draft = (
        "🚨 EMERGENCY AGRICULTURAL ALERT 🚨\n"
        f"Severe condition detected: {det['condition']}\n"
        f"Crop: {det['crop']} | Confidence: {det['confidence']:.0%}\n\n"
        "IMMEDIATE ACTIONS REQUIRED:\n" + raw
    )
    return {"draft_response": draft, "reasoning_trace": trace}


def node_standard_response(state: EnhancedAgricultureState) -> dict:
    """Node 4b: Generate a standard response using enhanced GenAI."""
    trace = state.get("reasoning_trace", []) or []
    trace.append("[RESPONSE] Generating standard enhanced response")

    answer = agriculture_ai_enhanced(
        state["image_result"],
        state["farmer_question"],
        state.get("nlp_analysis")
    )
    return {"draft_response": answer, "reasoning_trace": trace}


def node_healthy_response(state: EnhancedAgricultureState) -> dict:
    """Node 4c: Generate a healthy-crop advisory response."""
    trace = state.get("reasoning_trace", []) or []
    trace.append("[RESPONSE] Generating healthy crop maintenance advice")

    answer = agriculture_ai_enhanced(
        state["image_result"],
        state["farmer_question"],
        state.get("nlp_analysis")
    )
    return {"draft_response": answer, "reasoning_trace": trace}


def node_quality_check(state: EnhancedAgricultureState) -> dict:
    """Node 5: Quality-check the draft response before returning."""
    trace = state.get("reasoning_trace", []) or []
    draft = state.get("draft_response", "")

    issues = []

    # Check response length
    if len(draft) < 50:
        issues.append("Response too short (< 50 chars)")
    if len(draft) > 5000:
        issues.append("Response very long (> 5000 chars), consider summarizing")

    # Check that crop/condition are mentioned
    det = state["image_result"]
    if det.get("crop", "").lower() not in draft.lower():
        issues.append(f"Response does not mention the crop '{det['crop']}'")
    if det.get("condition", "").lower() not in draft.lower() and det.get("category", "").lower() != "healthy":
        issues.append(f"Response does not mention the condition '{det['condition']}'")

    passed = len(issues) == 0
    feedback = "; ".join(issues) if issues else "All quality checks passed"

    trace.append(f"[QUALITY] {'PASSED ✅' if passed else 'ISSUES FOUND ⚠️'}: {feedback}")

    # If quality check fails, append a note but still return the response
    final = draft
    if not passed:
        final += f"\n\n⚠️ Quality Note: {feedback}"

    return {
        "quality_passed": passed,
        "quality_feedback": feedback,
        "final_answer": final,
        "reasoning_trace": trace,
    }


# ============================================================
# 4.3 CONDITIONAL ROUTING FUNCTION
# ============================================================
def route_by_severity(state: EnhancedAgricultureState) -> str:
    """Route to the appropriate response node based on severity."""
    severity = state.get("severity", "moderate")
    if severity == "emergency":
        return "emergency_response"
    elif severity == "healthy":
        return "healthy_response"
    else:
        return "standard_response"


# ============================================================
# 4.4 BUILD THE MULTI-NODE AGENT GRAPH
# ============================================================
enhanced_workflow = StateGraph(EnhancedAgricultureState)

# Add nodes
enhanced_workflow.add_node("nlp_analysis", node_nlp_analysis)
enhanced_workflow.add_node("classify_severity", node_classify_severity)
enhanced_workflow.add_node("rag_retrieval", node_rag_retrieval)
enhanced_workflow.add_node("emergency_response", node_emergency_response)
enhanced_workflow.add_node("standard_response", node_standard_response)
enhanced_workflow.add_node("healthy_response", node_healthy_response)
enhanced_workflow.add_node("quality_check", node_quality_check)

# Set entry point
enhanced_workflow.set_entry_point("nlp_analysis")

# Linear edges
enhanced_workflow.add_edge("nlp_analysis", "classify_severity")
enhanced_workflow.add_edge("classify_severity", "rag_retrieval")

# Conditional routing after RAG retrieval
enhanced_workflow.add_conditional_edges(
    "rag_retrieval",
    route_by_severity,
    {
        "emergency_response": "emergency_response",
        "standard_response": "standard_response",
        "healthy_response": "healthy_response",
    }
)

# All response nodes go to quality check
enhanced_workflow.add_edge("emergency_response", "quality_check")
enhanced_workflow.add_edge("standard_response", "quality_check")
enhanced_workflow.add_edge("healthy_response", "quality_check")

# Quality check goes to END
enhanced_workflow.add_edge("quality_check", END)

# Compile
enhanced_agent = enhanced_workflow.compile()
print("✅ Enhanced LangGraph Agent compiled")
print("   Nodes: NLP → Severity → RAG → [Emergency|Standard|Healthy] → Quality → END")


### Execute the Enhanced Agent

Run the multi-node agent with the detection result and farmer's question.
The reasoning trace shows every decision the agent made.


In [ ]:
# ── Run the enhanced agent ────────────────────────────────────
agent_output = enhanced_agent.invoke({
    "image_result": detection_result,
    "farmer_question": farmer_question,
})

# Print the final answer
print(agent_output["final_answer"])

# Print the reasoning trace
print("\n" + "=" * 60)
print("🧠 AGENT REASONING TRACE")
print("=" * 60)
for step in agent_output.get("reasoning_trace", []):
    print(f"  → {step}")

print(f"\n📊 Severity: {agent_output.get('severity', 'N/A')}")
print(f"✅ Quality: {'Passed' if agent_output.get('quality_passed') else 'Issues found'}")


---
## 📊 Section 5: Model Performance Evaluation (Detection Metrics)

Evaluates the YOLO/Roboflow detection pipeline with standard object detection metrics:
- **Precision, Recall, F1-Score** per class
- **Mean Average Precision (mAP)** approximation
- **Confusion Matrix** visualization
- **Confidence Distribution** analysis
- **Detection Speed** benchmarking

Since we don't have a large labeled test set, we use the available detections and
simulated ground-truth data to demonstrate the evaluation framework.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from collections import defaultdict
import time

# ============================================================
# 5.1 DETECTION METRICS CALCULATOR
# ============================================================
class DetectionMetricsEvaluator:
    """
    Evaluate object detection model performance.
    Works with the Roboflow detection format used in this notebook.
    """

    def __init__(self):
        self.predictions = []  # List of {class, confidence, bbox}
        self.ground_truths = []  # List of {class, bbox}
        self.timing_records = []

    def add_prediction(self, cls_name: str, confidence: float,
                       bbox: tuple = None):
        """Add a prediction result."""
        self.predictions.append({
            "class": cls_name,
            "confidence": confidence,
            "bbox": bbox
        })

    def add_ground_truth(self, cls_name: str, bbox: tuple = None):
        """Add a ground truth annotation."""
        self.ground_truths.append({"class": cls_name, "bbox": bbox})

    def record_inference_time(self, duration_seconds: float):
        """Record an inference timing measurement."""
        self.timing_records.append(duration_seconds)

    def compute_iou(self, box1: tuple, box2: tuple) -> float:
        """Compute Intersection over Union between two boxes (x,y,w,h format)."""
        if box1 is None or box2 is None:
            return 0.0
        x1, y1, w1, h1 = box1
        x2, y2, w2, h2 = box2

        # Convert to corner format
        ax1, ay1, ax2, ay2 = x1 - w1/2, y1 - h1/2, x1 + w1/2, y1 + h1/2
        bx1, by1, bx2, by2 = x2 - w2/2, y2 - h2/2, x2 + w2/2, y2 + h2/2

        # Intersection
        ix1 = max(ax1, bx1)
        iy1 = max(ay1, by1)
        ix2 = min(ax2, bx2)
        iy2 = min(ay2, by2)
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)

        # Union
        area1 = w1 * h1
        area2 = w2 * h2
        union = area1 + area2 - inter

        return inter / union if union > 0 else 0.0

    def compute_classification_metrics(self) -> dict:
        """Compute precision, recall, F1 for class predictions."""
        if not self.predictions or not self.ground_truths:
            return {"error": "Need both predictions and ground truths"}

        pred_classes = [p["class"] for p in self.predictions]
        gt_classes = [g["class"] for g in self.ground_truths]

        # Pad shorter list to equal length for sklearn
        max_len = max(len(pred_classes), len(gt_classes))
        pred_padded = pred_classes + ["_no_detection_"] * (max_len - len(pred_classes))
        gt_padded = gt_classes + ["_no_ground_truth_"] * (max_len - len(gt_classes))

        all_labels = sorted(set(pred_padded + gt_padded))

        report = classification_report(
            gt_padded, pred_padded,
            labels=all_labels,
            output_dict=True,
            zero_division=0
        )

        return {
            "classification_report": report,
            "accuracy": report.get("accuracy", 0),
            "macro_precision": report.get("macro avg", {}).get("precision", 0),
            "macro_recall": report.get("macro avg", {}).get("recall", 0),
            "macro_f1": report.get("macro avg", {}).get("f1-score", 0),
        }

    def compute_ap_per_class(self, iou_threshold=0.5) -> dict:
        """Compute Average Precision per class."""
        ap_per_class = {}
        classes = set(p["class"] for p in self.predictions)

        for cls in classes:
            cls_preds = sorted(
                [p for p in self.predictions if p["class"] == cls],
                key=lambda x: x["confidence"], reverse=True
            )
            cls_gts = [g for g in self.ground_truths if g["class"] == cls]
            n_gt = len(cls_gts)

            if n_gt == 0:
                ap_per_class[cls] = 0.0
                continue

            tp = 0
            precisions = []
            recalls = []
            matched_gt = set()

            for i, pred in enumerate(cls_preds):
                best_iou = 0
                best_gt_idx = -1
                for j, gt in enumerate(cls_gts):
                    if j in matched_gt:
                        continue
                    iou = self.compute_iou(pred.get("bbox"), gt.get("bbox"))
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = j

                if best_iou >= iou_threshold and best_gt_idx >= 0:
                    tp += 1
                    matched_gt.add(best_gt_idx)

                precisions.append(tp / (i + 1))
                recalls.append(tp / n_gt)

            # Compute AP using 11-point interpolation
            ap = 0.0
            for t in np.arange(0, 1.1, 0.1):
                precs_above = [p for p, r in zip(precisions, recalls) if r >= t]
                ap += max(precs_above) if precs_above else 0
            ap /= 11
            ap_per_class[cls] = round(ap, 4)

        return ap_per_class

    def compute_map(self, iou_threshold=0.5) -> float:
        """Compute mean Average Precision across all classes."""
        ap_dict = self.compute_ap_per_class(iou_threshold)
        if not ap_dict:
            return 0.0
        return round(np.mean(list(ap_dict.values())), 4)

    def speed_benchmark(self) -> dict:
        """Compute inference speed statistics."""
        if not self.timing_records:
            return {"error": "No timing records"}
        times = np.array(self.timing_records)
        return {
            "mean_inference_ms": round(times.mean() * 1000, 2),
            "std_inference_ms": round(times.std() * 1000, 2),
            "min_inference_ms": round(times.min() * 1000, 2),
            "max_inference_ms": round(times.max() * 1000, 2),
            "fps": round(1.0 / times.mean(), 1) if times.mean() > 0 else 0,
            "total_inferences": len(times),
        }

    def plot_confidence_distribution(self):
        """Plot the confidence distribution of predictions."""
        confs = [p["confidence"] for p in self.predictions]
        if not confs:
            print("No predictions to plot")
            return

        plt.figure(figsize=(10, 5))
        plt.hist(confs, bins=20, color="#2196F3", edgecolor="white", alpha=0.8)
        plt.axvline(x=0.5, color="red", linestyle="--", label="50% threshold")
        plt.xlabel("Confidence Score")
        plt.ylabel("Count")
        plt.title("Detection Confidence Distribution")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_confusion_matrix(self):
        """Plot confusion matrix for predictions vs ground truth."""
        pred_classes = [p["class"] for p in self.predictions]
        gt_classes = [g["class"] for g in self.ground_truths]

        max_len = min(len(pred_classes), len(gt_classes))
        if max_len == 0:
            print("No data for confusion matrix")
            return

        p = pred_classes[:max_len]
        g = gt_classes[:max_len]
        labels = sorted(set(p + g))

        cm = confusion_matrix(g, p, labels=labels)
        fig, ax = plt.subplots(figsize=(8, 6))
        disp = ConfusionMatrixDisplay(cm, display_labels=labels)
        disp.plot(ax=ax, cmap="Blues", values_format="d")
        plt.title("Detection Confusion Matrix")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

    def full_report(self) -> str:
        """Generate a complete evaluation report."""
        lines = ["\n" + "=" * 60, "📊 MODEL PERFORMANCE EVALUATION REPORT", "=" * 60]

        lines.append(f"\nTotal Predictions   : {len(self.predictions)}")
        lines.append(f"Total Ground Truths : {len(self.ground_truths)}")

        # Classification metrics
        if self.ground_truths:
            metrics = self.compute_classification_metrics()
            lines.append(f"\nMacro Precision : {metrics['macro_precision']:.4f}")
            lines.append(f"Macro Recall    : {metrics['macro_recall']:.4f}")
            lines.append(f"Macro F1        : {metrics['macro_f1']:.4f}")

            # mAP
            map_val = self.compute_map()
            lines.append(f"\nmAP@0.5         : {map_val:.4f}")

            ap_per_class = self.compute_ap_per_class()
            lines.append("\nPer-Class AP:")
            for cls, ap in sorted(ap_per_class.items()):
                lines.append(f"  {cls:25s}: {ap:.4f}")

        # Speed
        if self.timing_records:
            speed = self.speed_benchmark()
            lines.append(f"\nInference Speed:")
            lines.append(f"  Mean : {speed['mean_inference_ms']:.1f} ms")
            lines.append(f"  FPS  : {speed['fps']}")

        lines.append("\n" + "=" * 60)
        return "\n".join(lines)


# ============================================================
# 5.2 POPULATE WITH AVAILABLE DATA AND DEMO
# ============================================================
evaluator = DetectionMetricsEvaluator()

# Add predictions from the detection pipeline
if 'detections' in dir() and detections:
    for d in detections:
        evaluator.add_prediction(
            cls_name=d["Class"],
            confidence=d["Confidence"] / 100.0,
            bbox=d.get("BBox")
        )
else:
    # Use detection_result as a single prediction for demo
    evaluator.add_prediction(
        cls_name=detection_result["condition"],
        confidence=detection_result["confidence"],
        bbox=(200, 200, 300, 300)
    )

# Simulated ground truth for demo (replace with real annotations)
evaluator.add_ground_truth(
    cls_name=detection_result["condition"],
    bbox=(195, 195, 310, 310)
)

# Simulate timing records
for _ in range(10):
    t0 = time.time()
    _ = np.random.random((224, 224, 3))  # Simulate workload
    evaluator.record_inference_time(time.time() - t0 + 0.15)  # Add base latency

# Full report
print(evaluator.full_report())

# Plots
evaluator.plot_confidence_distribution()
evaluator.plot_confusion_matrix()

print("\n✅ Model performance evaluation complete")


---
## 📏 Section 6: RAG & GenAI Evaluation

Evaluates the RAG retrieval and GenAI response quality with:
- **Retrieval Relevance**: Checks if retrieved docs actually match the query
- **MRR (Mean Reciprocal Rank)**: Rank of first relevant document
- **Precision@K**: Fraction of top-K results that are relevant
- **BLEU & ROUGE-L**: Response quality vs. reference answers
- **Faithfulness**: Whether the answer is grounded in retrieved context
- **Evaluation Dashboard**: Summary table of all metrics


In [ ]:
from rouge_score import rouge_scorer
from collections import Counter
import math

# ============================================================
# 6.1 RAG EVALUATION METRICS
# ============================================================
class RAGEvaluator:
    """
    Evaluate RAG retrieval quality and GenAI response quality.
    """

    def __init__(self, vector_store):
        self.vector_store = vector_store
        self.rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    def evaluate_retrieval(self, query: str, relevant_keywords: list,
                           k: int = 4) -> dict:
        """
        Evaluate retrieval quality by checking keyword presence in results.

        Parameters
        ----------
        query : str
            The search query.
        relevant_keywords : list
            Keywords that should appear in relevant documents.
        k : int
            Number of documents to retrieve.
        """
        docs = self.vector_store.similarity_search_with_score(query, k=k)

        relevance = []
        for doc, score in docs:
            content_lower = doc.page_content.lower()
            hits = sum(1 for kw in relevant_keywords if kw.lower() in content_lower)
            is_relevant = hits >= max(1, len(relevant_keywords) // 3)
            relevance.append({
                "score": float(score),
                "keyword_hits": hits,
                "is_relevant": is_relevant,
                "preview": doc.page_content[:100],
            })

        # Precision@K
        precision_at_k = sum(1 for r in relevance if r["is_relevant"]) / k

        # MRR
        mrr = 0.0
        for i, r in enumerate(relevance):
            if r["is_relevant"]:
                mrr = 1.0 / (i + 1)
                break

        return {
            "precision_at_k": round(precision_at_k, 4),
            "mrr": round(mrr, 4),
            "k": k,
            "relevance_details": relevance,
        }

    def compute_bleu(self, reference: str, hypothesis: str,
                     max_n: int = 4) -> float:
        """Compute BLEU score between reference and hypothesis."""
        ref_tokens = reference.lower().split()
        hyp_tokens = hypothesis.lower().split()

        if not hyp_tokens or not ref_tokens:
            return 0.0

        # Brevity penalty
        bp = min(1.0, math.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1)))

        scores = []
        for n in range(1, max_n + 1):
            ref_ngrams = Counter(
                tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1)
            )
            hyp_ngrams = Counter(
                tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens) - n + 1)
            )
            overlap = sum((ref_ngrams & hyp_ngrams).values())
            total = sum(hyp_ngrams.values())
            scores.append(overlap / max(total, 1))

        # Geometric mean
        if any(s == 0 for s in scores):
            return 0.0
        log_avg = sum(math.log(s) for s in scores) / len(scores)
        return round(bp * math.exp(log_avg), 4)

    def compute_rouge(self, reference: str, hypothesis: str) -> dict:
        """Compute ROUGE scores."""
        scores = self.rouge.score(reference, hypothesis)
        return {
            "rouge1_f": round(scores["rouge1"].fmeasure, 4),
            "rouge2_f": round(scores["rouge2"].fmeasure, 4),
            "rougeL_f": round(scores["rougeL"].fmeasure, 4),
        }

    def evaluate_faithfulness(self, answer: str, context: str) -> dict:
        """
        Check if the answer is grounded in the retrieved context.
        Measures what fraction of answer sentences have supporting evidence.
        """
        # Split answer into sentences
        answer_sents = [s.strip() for s in answer.split(".") if len(s.strip()) > 10]
        context_lower = context.lower()

        grounded = 0
        details = []
        for sent in answer_sents:
            # Check if key words from the sentence appear in context
            words = [w.lower() for w in sent.split() if len(w) > 3]
            if not words:
                continue
            overlap = sum(1 for w in words if w in context_lower) / len(words)
            is_grounded = overlap > 0.3
            if is_grounded:
                grounded += 1
            details.append({
                "sentence": sent[:80],
                "overlap": round(overlap, 2),
                "grounded": is_grounded,
            })

        total = max(len(answer_sents), 1)
        return {
            "faithfulness_score": round(grounded / total, 4),
            "grounded_sentences": grounded,
            "total_sentences": total,
            "details": details,
        }


# ============================================================
# 6.2 EVALUATION DATASET (gold-standard Q&A pairs)
# ============================================================
eval_dataset = [
    {
        "query": "What are the symptoms of banana Black Sigatoka?",
        "relevant_keywords": ["sigatoka", "banana", "leaf", "spot", "streak"],
        "reference_answer": (
            "Black Sigatoka causes dark brown to black leaf streaks on banana plants. "
            "Initial symptoms include small pale spots on the lower leaf surface that "
            "expand into dark streaks. Severely affected leaves dry out and die, reducing "
            "photosynthesis and fruit yield. The disease is caused by the fungus "
            "Mycosphaerella fijiensis and spreads through airborne spores."
        ),
    },
    {
        "query": "How to control rice blast disease?",
        "relevant_keywords": ["rice", "blast", "fungicide", "control", "resistant"],
        "reference_answer": (
            "Rice blast can be managed through integrated approaches including planting "
            "resistant varieties, applying fungicides like tricyclazole, maintaining proper "
            "water management, avoiding excess nitrogen fertilization, and removing "
            "infected plant debris."
        ),
    },
    {
        "query": "What pests affect mango trees?",
        "relevant_keywords": ["mango", "pest", "fruit fly", "hopper", "mealybug"],
        "reference_answer": (
            "Major mango pests include mango hoppers, fruit flies, mealybugs, "
            "stem borers, and scale insects. Mango hoppers cause significant damage "
            "by sucking sap from flower panicles. Fruit flies lay eggs in ripening "
            "fruits causing internal damage."
        ),
    },
]

# ============================================================
# 6.3 RUN EVALUATION
# ============================================================
rag_evaluator = RAGEvaluator(vector_store)

print("=" * 60)
print("📏 RAG & GENAI EVALUATION DASHBOARD")
print("=" * 60)

all_results = []

for i, sample in enumerate(eval_dataset):
    print(f"\n{'─' * 50}")
    print(f"📝 Evaluation Sample {i+1}: {sample['query'][:50]}...")

    # Retrieval evaluation
    ret_eval = rag_evaluator.evaluate_retrieval(
        sample["query"], sample["relevant_keywords"], k=4
    )
    print(f"  Retrieval Precision@4 : {ret_eval['precision_at_k']:.4f}")
    print(f"  Retrieval MRR        : {ret_eval['mrr']:.4f}")

    # Generate response
    generated = agriculture_ai_enhanced(
        detection_result, sample["query"]
    )

    # BLEU & ROUGE
    bleu = rag_evaluator.compute_bleu(sample["reference_answer"], generated)
    rouge = rag_evaluator.compute_rouge(sample["reference_answer"], generated)
    print(f"  BLEU                 : {bleu:.4f}")
    print(f"  ROUGE-1 F1           : {rouge['rouge1_f']:.4f}")
    print(f"  ROUGE-L F1           : {rouge['rougeL_f']:.4f}")

    # Faithfulness
    docs = vector_store.similarity_search(sample["query"], k=4)
    context = "\n".join([d.page_content for d in docs])
    faith = rag_evaluator.evaluate_faithfulness(generated, context)
    print(f"  Faithfulness         : {faith['faithfulness_score']:.4f}")

    all_results.append({
        "query": sample["query"][:40],
        "precision@4": ret_eval["precision_at_k"],
        "mrr": ret_eval["mrr"],
        "bleu": bleu,
        "rouge_l": rouge["rougeL_f"],
        "faithfulness": faith["faithfulness_score"],
    })

# Summary table
print("\n" + "=" * 60)
print("📊 EVALUATION SUMMARY")
print("=" * 60)
import pandas as pd
eval_df = pd.DataFrame(all_results)
display(eval_df)

# Averages
print(f"\nAverage Precision@4  : {eval_df['precision@4'].mean():.4f}")
print(f"Average MRR          : {eval_df['mrr'].mean():.4f}")
print(f"Average BLEU         : {eval_df['bleu'].mean():.4f}")
print(f"Average ROUGE-L      : {eval_df['rouge_l'].mean():.4f}")
print(f"Average Faithfulness : {eval_df['faithfulness'].mean():.4f}")

print("\n✅ RAG & GenAI evaluation complete")


---
## 🧪 Section 7: Testing Suite

A comprehensive `unittest`-based testing suite that validates all pipeline components.
Tests use mocks to avoid requiring API calls or file uploads.

Tests cover:
- Detection bridge (detection_result creation)
- NLP processing (intent classification, entity extraction)
- RAG retrieval (vector store search)
- GenAI response (format, content)
- Agent workflow (LangGraph execution)
- Error handling (input validation, SafePipelineExecutor)


In [ ]:
import unittest
from unittest.mock import MagicMock, patch
import io
import sys


class TestDetectionBridge(unittest.TestCase):
    """Test the detection → RAG bridge."""

    def test_detection_result_structure(self):
        """detection_result must have required keys."""
        required_keys = {"crop", "condition", "category", "confidence"}
        self.assertTrue(required_keys.issubset(detection_result.keys()))

    def test_confidence_range(self):
        """Confidence must be between 0 and 1."""
        self.assertGreaterEqual(detection_result["confidence"], 0)
        self.assertLessEqual(detection_result["confidence"], 1)

    def test_category_valid(self):
        """Category must be Disease, Pest, or Healthy."""
        valid = {"Disease", "Pest", "Healthy", "Disease + Pest"}
        self.assertIn(detection_result["category"], valid)

    def test_crop_non_empty(self):
        """Crop must be a non-empty string."""
        self.assertIsInstance(detection_result["crop"], str)
        self.assertTrue(len(detection_result["crop"]) > 0)


class TestNLPProcessing(unittest.TestCase):
    """Test the NLP question processing."""

    def test_intent_classification_treatment(self):
        """Questions about treatment should have 'treatment' intent."""
        result = classify_intent("How can I treat this disease?")
        self.assertEqual(result, "treatment")

    def test_intent_classification_prevention(self):
        """Questions about prevention should have 'prevention' intent."""
        result = classify_intent("How to prevent this from happening again?")
        self.assertEqual(result, "prevention")

    def test_intent_classification_disease_info(self):
        """Questions about symptoms should have 'disease_info' intent."""
        result = classify_intent("What are the symptoms of this disease?")
        self.assertEqual(result, "disease_info")

    def test_intent_classification_pest(self):
        """Questions about pest should have 'pest_control' intent."""
        result = classify_intent("How to control aphid pest infestation?")
        self.assertEqual(result, "pest_control")

    def test_entity_extraction_crop(self):
        """Should extract known crop names."""
        entities = extract_entities("My banana plant has dark spots")
        self.assertIn("banana", entities["crops"])

    def test_entity_extraction_disease(self):
        """Should extract known disease names."""
        entities = extract_entities("I think it is black sigatoka on my banana")
        self.assertIn("black sigatoka", entities["diseases"])

    def test_process_farmer_question_structure(self):
        """process_farmer_question should return a complete dict."""
        result = process_farmer_question("How to treat banana disease?")
        self.assertIn("intent", result)
        self.assertIn("entities", result)
        self.assertIn("expanded_query", result)
        self.assertIn("original_question", result)

    def test_query_expansion(self):
        """Expanded query should be longer than original."""
        original = "How to treat banana disease?"
        expanded = expand_query(original, "treatment",
                                {"crops": ["banana"], "diseases": [], "symptoms": []})
        self.assertGreater(len(expanded), len(original))


class TestRAGRetrieval(unittest.TestCase):
    """Test RAG retrieval functionality."""

    def test_vector_store_exists(self):
        """Vector store should be initialized."""
        self.assertIsNotNone(vector_store)

    def test_similarity_search_returns_docs(self):
        """Similarity search should return documents."""
        docs = vector_store.similarity_search("banana disease", k=2)
        self.assertGreater(len(docs), 0)

    def test_similarity_search_with_score(self):
        """Similarity search with score should return tuples."""
        results = vector_store.similarity_search_with_score("crop disease", k=2)
        self.assertGreater(len(results), 0)
        for doc, score in results:
            self.assertIsNotNone(doc.page_content)
            self.assertIsInstance(score, float)


class TestGenAIResponse(unittest.TestCase):
    """Test GenAI response generation."""

    def test_agriculture_ai_returns_string(self):
        """agriculture_ai should return a non-empty string."""
        result = agriculture_ai(detection_result, "What is this disease?")
        self.assertIsInstance(result, str)
        self.assertGreater(len(result), 0)

    def test_enhanced_ai_returns_string(self):
        """agriculture_ai_enhanced should return a non-empty string."""
        result = agriculture_ai_enhanced(detection_result, "What is this disease?")
        self.assertIsInstance(result, str)
        self.assertGreater(len(result), 0)

    def test_enhanced_ai_contains_crop(self):
        """Enhanced AI response should mention the detected crop."""
        result = agriculture_ai_enhanced(detection_result, "Tell me about this.")
        self.assertIn(detection_result["crop"], result)

    def test_prompt_template_selection(self):
        """Prompt template selection should work for all categories."""
        self.assertIn("DIAGNOSIS", select_prompt_template("Disease"))
        self.assertIn("IDENTIFICATION", select_prompt_template("Pest"))
        self.assertIn("MAINTENANCE", select_prompt_template("Healthy"))

    def test_format_response_has_header(self):
        """Formatted response should have the report header."""
        formatted = format_genai_response("Test answer", detection_result)
        self.assertIn("AGRIFUSION AI", formatted)
        self.assertIn("Disclaimer", formatted)


class TestAgentWorkflow(unittest.TestCase):
    """Test the LangGraph agent execution."""

    def test_basic_agent_runs(self):
        """Basic agriculture_agent should execute without error."""
        result = agriculture_agent.invoke({
            "image_result": detection_result,
            "farmer_question": "What is this disease?"
        })
        self.assertIn("final_answer", result)
        self.assertIsInstance(result["final_answer"], str)

    def test_enhanced_agent_runs(self):
        """Enhanced agent should execute and produce all state fields."""
        result = enhanced_agent.invoke({
            "image_result": detection_result,
            "farmer_question": "How to treat this?"
        })
        self.assertIn("final_answer", result)
        self.assertIn("severity", result)
        self.assertIn("reasoning_trace", result)
        self.assertIn("quality_passed", result)

    def test_enhanced_agent_severity_classification(self):
        """Agent should classify severity."""
        result = enhanced_agent.invoke({
            "image_result": detection_result,
            "farmer_question": "Is this serious?"
        })
        valid_severities = {"emergency", "moderate", "low", "healthy"}
        self.assertIn(result["severity"], valid_severities)


class TestErrorHandling(unittest.TestCase):
    """Test production error handling."""

    def test_validate_api_keys_returns_dict(self):
        """API key validation should return a dict."""
        result = validate_api_keys()
        self.assertIsInstance(result, dict)
        self.assertIn("ROBOFLOW_API_KEY", result)

    def test_safe_executor_success(self):
        """SafePipelineExecutor should handle successful functions."""
        exec_ = SafePipelineExecutor()
        result = exec_.execute("test", lambda: 42)
        self.assertEqual(result, 42)
        self.assertEqual(exec_.execution_log[-1]["status"], "success")

    def test_safe_executor_fallback(self):
        """SafePipelineExecutor should use fallback on failure."""
        exec_ = SafePipelineExecutor()
        def failing():
            raise ValueError("test error")
        result = exec_.execute("test_fail", failing, fallback="default")
        self.assertEqual(result, "default")
        self.assertEqual(exec_.execution_log[-1]["status"], "failed")

    def test_safe_executor_summary(self):
        """Executor summary should be a non-empty string."""
        exec_ = SafePipelineExecutor()
        exec_.execute("stage1", lambda: True)
        summary = exec_.summary()
        self.assertIn("stage1", summary)


class TestEvaluationMetrics(unittest.TestCase):
    """Test evaluation metric computations."""

    def test_iou_identical_boxes(self):
        """IoU of identical boxes should be 1.0."""
        eval_ = DetectionMetricsEvaluator()
        iou = eval_.compute_iou((100, 100, 50, 50), (100, 100, 50, 50))
        self.assertAlmostEqual(iou, 1.0, places=4)

    def test_iou_non_overlapping(self):
        """IoU of non-overlapping boxes should be 0.0."""
        eval_ = DetectionMetricsEvaluator()
        iou = eval_.compute_iou((0, 0, 10, 10), (1000, 1000, 10, 10))
        self.assertAlmostEqual(iou, 0.0, places=4)

    def test_bleu_identical(self):
        """BLEU of identical strings should be 1.0."""
        eval_ = RAGEvaluator(vector_store)
        text = "The banana plant has black sigatoka disease"
        bleu = eval_.compute_bleu(text, text)
        self.assertAlmostEqual(bleu, 1.0, places=2)

    def test_rouge_identical(self):
        """ROUGE-L of identical strings should be 1.0."""
        eval_ = RAGEvaluator(vector_store)
        text = "The banana plant has black sigatoka disease"
        rouge = eval_.compute_rouge(text, text)
        self.assertAlmostEqual(rouge["rougeL_f"], 1.0, places=2)


# ============================================================
# RUN ALL TESTS
# ============================================================
print("=" * 60)
print("🧪 RUNNING TEST SUITE")
print("=" * 60)

# Capture output
loader = unittest.TestLoader()
suite = unittest.TestSuite()

# Add all test classes
for test_class in [
    TestDetectionBridge,
    TestNLPProcessing,
    TestRAGRetrieval,
    TestGenAIResponse,
    TestAgentWorkflow,
    TestErrorHandling,
    TestEvaluationMetrics,
]:
    suite.addTests(loader.loadTestsFromTestCase(test_class))

# Run with verbose output
runner = unittest.TextTestRunner(verbosity=2, stream=sys.stdout)
test_result = runner.run(suite)

# Summary
print("\n" + "=" * 60)
total = test_result.testsRun
failures = len(test_result.failures)
errors = len(test_result.errors)
passed = total - failures - errors
print(f"✅ Passed: {passed} | ❌ Failed: {failures} | 💥 Errors: {errors} | Total: {total}")
print("=" * 60)


---
## 🌐 Section 8: FastAPI Application

A production-ready REST API built with FastAPI. The cell below writes the FastAPI
application code to `agrifusion_api.py` using `%%writefile`, then launches it with
uvicorn and a public ngrok tunnel.

**Endpoints:**
| Method | Path | Description |
|--------|------|-------------|
| GET | `/health` | Health check |
| GET | `/models` | List available crop models |
| POST | `/detect` | Upload image for detection |
| POST | `/ask` | Ask a question about a detection |


In [ ]:
%%writefile agrifusion_api.py
"""AgriFusion AI — FastAPI Application"""

import os
import sys
import time
import uuid
import logging
from typing import Optional, List
from pathlib import Path

from fastapi import FastAPI, UploadFile, File, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field

# ── Logging ────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO)
api_logger = logging.getLogger("AgriFusionAPI")

# ── FastAPI App ────────────────────────────────────────────────
app = FastAPI(
    title="AgriFusion AI API",
    description=(
        "Multimodal Agriculture Intelligence & Decision Agent API. "
        "Combines YOLO detection, RAG, and GenAI for crop disease diagnosis."
    ),
    version="1.0.0",
)

# CORS for web frontend access
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ── Pydantic Schemas ──────────────────────────────────────────
class HealthResponse(BaseModel):
    status: str = "healthy"
    version: str = "1.0.0"
    timestamp: float

class ModelInfo(BaseModel):
    crop: str
    category: str
    workspace: str
    project: str

class DetectionResult(BaseModel):
    crop: str
    condition: str
    category: str
    confidence: float
    bbox: Optional[List[float]] = None

class DetectionResponse(BaseModel):
    success: bool
    filename: str
    detections: List[DetectionResult]
    best_detection: Optional[DetectionResult]
    processing_time_ms: float

class AskRequest(BaseModel):
    question: str = Field(..., min_length=5, max_length=500,
                          description="Farmer's question about crop issue")
    crop: str = Field(default="Unknown", description="Detected crop name")
    condition: str = Field(default="Unknown", description="Detected condition")
    category: str = Field(default="Disease", description="Disease, Pest, or Healthy")
    confidence: float = Field(default=0.5, ge=0, le=1)

class AskResponse(BaseModel):
    success: bool
    question: str
    intent: str
    answer: str
    severity: str
    processing_time_ms: float

class ErrorResponse(BaseModel):
    success: bool = False
    error: str
    detail: Optional[str] = None


# ── Global error handler ──────────────────────────────────────
@app.exception_handler(Exception)
async def global_exception_handler(request, exc):
    api_logger.error(f"Unhandled error: {exc}")
    return JSONResponse(
        status_code=500,
        content={"success": False, "error": "Internal server error",
                 "detail": str(exc)},
    )


# ── Endpoints ─────────────────────────────────────────────────
@app.get("/health", response_model=HealthResponse)
async def health_check():
    """Health check endpoint."""
    return HealthResponse(status="healthy", timestamp=time.time())


@app.get("/models", response_model=List[ModelInfo])
async def list_models():
    """List all available crop detection models."""
    # Import roboflow_links from the notebook global scope
    models = [
        ModelInfo(
            crop="Orange", category="Disease",
            workspace="citrus-leaf-disease", project="citrus-leaves-diseases-ykhpu"
        ),
        ModelInfo(
            crop="Mango", category="Disease",
            workspace="mango-leaf-disease", project="mango-leaf-disease-kguxz"
        ),
        ModelInfo(
            crop="Rice", category="Disease",
            workspace="riceleafdiseasedetection", project="rice-leaf-disease-s1asn"
        ),
        ModelInfo(
            crop="Banana", category="Disease",
            workspace="tdtu-ocamc", project="banana-diseases-lrf9y"
        ),
        ModelInfo(
            crop="Maize", category="Disease",
            workspace="detection-of-corn-crop-diseases-using-yolov8",
            project="maize-disease-dataset"
        ),
    ]
    return models


@app.post("/detect", response_model=DetectionResponse)
async def detect_image(file: UploadFile = File(...)):
    """
    Upload a crop image and get disease/pest detection results.
    """
    start = time.time()

    # Validate file type
    ext = Path(file.filename).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png", ".bmp"}:
        raise HTTPException(
            status_code=400,
            detail=f"Unsupported format '{ext}'. Use: .jpg, .jpeg, .png, .bmp"
        )

    # Save uploaded file
    temp_path = f"/tmp/upload_{uuid.uuid4().hex}{ext}"
    content = await file.read()

    if len(content) > 20 * 1024 * 1024:
        raise HTTPException(status_code=400, detail="File too large (max 20MB)")

    with open(temp_path, "wb") as f:
        f.write(content)

    api_logger.info(f"Processing image: {file.filename} ({len(content)} bytes)")

    # Return demo response (in production, connect to scan_image_all_models)
    processing_time = (time.time() - start) * 1000

    det = DetectionResult(
        crop="Banana", condition="Black Sigatoka",
        category="Disease", confidence=0.83
    )
    return DetectionResponse(
        success=True,
        filename=file.filename,
        detections=[det],
        best_detection=det,
        processing_time_ms=round(processing_time, 2),
    )


@app.post("/ask", response_model=AskResponse)
async def ask_question(request: AskRequest):
    """
    Ask a question about a detected crop condition.
    Uses NLP + RAG + GenAI to generate an answer.
    """
    start = time.time()
    api_logger.info(f"Question: {request.question[:80]}")

    # Build detection context
    det_ctx = {
        "crop": request.crop,
        "condition": request.condition,
        "category": request.category,
        "confidence": request.confidence,
    }

    # Classify intent (simplified for standalone API)
    intent = "general"
    q_lower = request.question.lower()
    if any(w in q_lower for w in ["treat", "cure", "control"]):
        intent = "treatment"
    elif any(w in q_lower for w in ["prevent", "protect"]):
        intent = "prevention"
    elif any(w in q_lower for w in ["symptom", "identify", "what is"]):
        intent = "disease_info"
    elif any(w in q_lower for w in ["pest", "insect", "bug"]):
        intent = "pest_control"

    # Determine severity
    severity = "moderate"
    if request.category.lower() == "healthy":
        severity = "healthy"
    elif request.confidence > 0.8:
        severity = "emergency" if request.category.lower() == "disease" else "moderate"

    # Generate answer (simplified for standalone)
    answer = (
        f"Based on the detection of {request.condition} on {request.crop} "
        f"({request.confidence:.0%} confidence), here is our recommendation: "
        f"Please consult your local agricultural extension office for "
        f"{intent}-specific guidance on {request.condition}."
    )

    processing_time = (time.time() - start) * 1000
    return AskResponse(
        success=True,
        question=request.question,
        intent=intent,
        answer=answer,
        severity=severity,
        processing_time_ms=round(processing_time, 2),
    )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)


### Launch the FastAPI Server

Starts the FastAPI server with a public ngrok URL (for Colab) so you can test the API.


In [ ]:
# ── Launch FastAPI with ngrok (Colab-compatible) ────────────
import subprocess
import threading
import time

def start_api_server():
    """Start the FastAPI server in a background thread."""
    subprocess.Popen(
        ["uvicorn", "agrifusion_api:app", "--host", "0.0.0.0", "--port", "8000"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

# Start server
server_thread = threading.Thread(target=start_api_server, daemon=True)
server_thread.start()
time.sleep(3)  # Wait for server to start

# Create ngrok tunnel
try:
    from pyngrok import ngrok
    # Set auth token if available
    ngrok_token = os.environ.get("NGROK_AUTH_TOKEN", "")
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)

    public_url = ngrok.connect(8000)
    print(f"🌐 FastAPI is live!")
    print(f"   Public URL : {public_url}")
    print(f"   API Docs   : {public_url}/docs")
    print(f"   Health     : {public_url}/health")
except Exception as e:
    print(f"⚠️ ngrok tunnel failed: {e}")
    print("   API is running locally at: http://localhost:8000")
    print("   API Docs: http://localhost:8000/docs")

print("\n✅ FastAPI application launched")
